In [1]:
import time
import numpy as np
import random
import pandas as pd
from tqdm import tqdm
from multiprocessing import Process, Queue

In [2]:

def run_division_process(queue, numerator, denominator, sleep_sec):
    try:        
        # time.sleep(sleep_sec)
        result = numerator/denominator        
        queue.put(result)
        print("result", result, queue.get())
    except Exception as e:
        queue.put(e)

def division_with_timeout(numerator, denominator, timeout_sec):
    q = Queue()
    sleep_sec = random.randint(1, timeout_sec*2)
    print(timeout_sec, denominator, sleep_sec)
    p = Process(
        target=run_division_process,
        args=(q, numerator, denominator, sleep_sec)
    )
    p.start()
    p.join(timeout=timeout_sec)

    if p.is_alive():
        p.terminate()
        p.join()
        return None  # Timeout occurred

    result = q.get() if not q.empty() else None
    if isinstance(result, Exception):
        raise result
    return result

def check_division_with_timeout(numerator, denominators, timeout_sec=30*60):
    results = {}
    for d in denominators:
        results[d] = {"division": "fail"}
        try:
            result = division_with_timeout(numerator, d, timeout_sec)
            if result is None:
                print(f"⏰ Timeout (>{timeout_sec//60} min) for denominator '{d}' — skipping to next.")
                results[d]["division"] = "timeout"
                continue

            results[d]["division"] = "success"

        except Exception as e:
            print(f'Error calculating quotient for {d}')
            print(f'{e}')
            results[d]["division"] = "Error"
            
        print('division: ', results[d]["division"])

    return results


In [3]:

numerator = 20
denominators = [1,2,3,4,6,7,8,9,0]
check_division_with_timeout(numerator,denominators,10)


10 1 10
⏰ Timeout (>0 min) for denominator '1' — skipping to next.
10 2 15
⏰ Timeout (>0 min) for denominator '2' — skipping to next.
10 3 17
⏰ Timeout (>0 min) for denominator '3' — skipping to next.
10 4 13
⏰ Timeout (>0 min) for denominator '4' — skipping to next.
10 6 7
⏰ Timeout (>0 min) for denominator '6' — skipping to next.
10 7 11
⏰ Timeout (>0 min) for denominator '7' — skipping to next.
10 8 4
⏰ Timeout (>0 min) for denominator '8' — skipping to next.
10 9 4
⏰ Timeout (>0 min) for denominator '9' — skipping to next.
10 0 17
⏰ Timeout (>0 min) for denominator '0' — skipping to next.


{1: {'division': 'timeout'},
 2: {'division': 'timeout'},
 3: {'division': 'timeout'},
 4: {'division': 'timeout'},
 6: {'division': 'timeout'},
 7: {'division': 'timeout'},
 8: {'division': 'timeout'},
 9: {'division': 'timeout'},
 0: {'division': 'timeout'}}

In [8]:
import multiprocessing
import time
from queue import Empty

def long_running_function(return_queue, n):
    """The function to be executed with a timeout."""
    print("aa")
    try:
        print("bb")
        print(f"Function started with argument {n}")
        # Simulate work that might take too long
        time.sleep(n)
        result = f"Finished in {n} seconds"
        return_queue.put(result) # Put the result into the queue
    except Exception as e:
        print("cc")
        # Handle exceptions within the worker process
        return_queue.put(e)

def run_with_timeout(func, args, timeout):
    """Executes a function in a separate process with a timeout."""
    print("timeout", timeout)
    return_queue = multiprocessing.Queue()
    # Create a process targeting the function with its arguments
    p = multiprocessing.Process(target=func, args=(return_queue, *args))
    p.start()

    try:
        # Wait for the process to finish or for the timeout to expire
        print("a")
        p.join(timeout)
        print("b")
        if p.is_alive():
            # Process is still alive, so it timed out
            print("c")
            p.terminate()
            print("d")
            p.join() # Clean up the terminated process
            print("e")
            print(f"Function call timed out after {timeout} seconds")            
            raise TimeoutError(f"Function call timed out after {timeout} seconds")
        print("f")
        # If the process finished, try to get the result from the queue
        # print("return_queue.get_nowait()", return_queue.get_nowait())
        return return_queue.get() if not return_queue.empty() else None
        return return_queue.get_nowait()
        
    except Empty:
        # This could happen if the process finished but didn't put a result
        print("Empty")
        return None 
    except TimeoutError as e:
        print("raise e")
        raise e
    except Exception as e:
        # Re-raise any other exception from the worker process
        print("Exception as e")
        if not return_queue.empty():
            worker_exception = return_queue.get_nowait()
            if isinstance(worker_exception, Exception):
                raise worker_exception
        raise e



In [9]:
# 1. Success case: finishes within the limit
TIMEOUT_LIMIT = 5
try:
    result_success = run_with_timeout(long_running_function, (3,), TIMEOUT_LIMIT)
    print(f"\nResult (Success): {result_success}")
except TimeoutError as e:
    print(f"\nTimeout Caught: {e}")

print("-" * 20)

timeout 5
a
b
f

Result (Success): None
--------------------


In [55]:
result_success

In [7]:

# 2. Timeout case: exceeds the limit
try:
    result_timeout = run_with_timeout(long_running_function, (7,), TIMEOUT_LIMIT)
    print(f"\nResult (Timeout): {result_timeout}")
except TimeoutError as e:
    print(f"\nTimeout Caught: {e}")

a
b
f

Result (Timeout): None


In [10]:
import signal

class TimeoutException(Exception):   # Custom exception class
    pass

def timeout_handler(signum, frame):   # Custom signal handler
    raise TimeoutException

# Change the behavior of SIGALRM
signal.signal(signal.SIGALRM, timeout_handler)

for i in range(3):
    # Start the timer. Once 5 seconds are over, a SIGALRM signal is sent.
    signal.alarm(5)    
    # This try/except loop ensures that 
    #   you'll catch TimeoutException when it's sent.
    try:
        A(i) # Whatever your function that might hang
    except TimeoutException:
        continue # continue the for loop if function A takes more than 5 second
    else:
        # Reset the alarm
        signal.alarm(0)

AttributeError: module 'signal' has no attribute 'SIGALRM'

In [13]:
import time
import signal

class TimeoutException(Exception):   # Custom exception class
    pass


def break_after(seconds=2):
    def timeout_handler(signum, frame):   # Custom signal handler
        raise TimeoutException
    def function(function):
        def wrapper(*args, **kwargs):
            signal.signal(signal.SIGALRM, timeout_handler)
            signal.alarm(seconds)
            try:
                res = function(*args, **kwargs)
                signal.alarm(0)      # Clear alarm
                return res
            except TimeoutException:
                print('Oops, timeout: %s sec reached.' % seconds, function.__name__, args, kwargs)
            return
        return wrapper
    return function

In [14]:
@break_after(3)
def test(a, b, c):
    return time.sleep(10)

test(1,2,3)

AttributeError: module 'signal' has no attribute 'SIGALRM'

In [17]:
def bar():
    for i in range(100):
        print("Tick")
        time.sleep(1)

# Start bar as a process
p = multiprocessing.Process(target=bar)
p.start()

# Wait for 10 seconds or until process finishes
p.join(3)

# If thread is still active
if p.is_alive():
    print("running... let's kill it...")

    # Terminate - may not work if process is stuck for good
    p.terminate()
    # OR Kill - will work for sure, no chance for process to finish nicely however
    # p.kill()

    p.join()